# DAIS 2026 — Deployment & Setup

Short checklist to get a workspace ready for the DAIS 2026 runbooks
(`Genie.ipynb`, `AgentBricks.ipynb`, `LakebaseApps.ipynb`, `MLflow.ipynb`).
All four run against a single `all`-target deployment.

### 1. Create Domains

In **Catalog → Discover → Domains → New domain**, create three domains and select the matching existing governed tag for each:

| Domain | Governed tag (select existing) |
|---|---|
| **Operations** | `caspers_domain_operations = true` |
| **Revenue & Customers** | `caspers_domain_revenue = true` |
| **Compliance & Safety** | `caspers_domain_compliance = true` |

### 2. Enable Data quality monitor.
Enable schema-level anomaly detection on `${CATALOG}.food_safety`

### 3. Add prompts schema to the experiments in MAS and others
schema <Catalog>.prompts

### 4. Fix agent endpoint UC perms


In [ ]:
# EDIT THIS to match the catalog you passed to `bundle deploy --var catalog=...`
# (bundle default is `caspersdev`; this workspace uses something else if you
#  deployed with `--var catalog=<other>`).
CATALOG = ""  # e.g. "caspersdev"

assert CATALOG, "Set CATALOG above to the catalog name from your last bundle deploy."

result = dbutils.notebook.run(
    "../../utils/fix_agent_perms",
    1800,
    {
        "CATALOG": CATALOG,
        # Leave ENDPOINT_NAMES empty to use the default
        # f"{CATALOG}_{refund,complaint,support}_agent" pattern.
        "ENDPOINT_NAMES": "",
        "MAX_WAIT_SECONDS": "300",
    },
)
print(result)

### 5. Set up the Unity AI Gateway endpoint (`all` target only)

Both the Refund agent (LangGraph + `databricks_langchain.ChatDatabricks`)
and the Complaint agent (DSPy + `dspy.LM('openai/...')`) on the `all`
target route **every** internal LLM call through a single Unity AI Gateway
(v2 Beta) endpoint, so every agent LLM call gets PII guardrails,
inference-table audit, usage tracking, and rate limits applied centrally —
and you can attribute usage to either agent via the `requester` column on
the inference table.

Both agents are deployed as **Databricks Apps** (`apps/refund-agent`,
`apps/complaint-agent`) running the MLflow `AgentServer`, not as Model
Serving endpoints. Gateway routing is **always on** — there is no
foundation-model fallback. Each App reads the gateway endpoint name from
its `AI_GATEWAY_ENDPOINT_NAME` env var (set from the job parameter of the
same name at deploy time) and POSTs to `<host>/ai-gateway/mlflow/v1` with
that name as the request `model`.

The v2 Beta gateway is **UI-configured only** — there is no public REST /
SDK API for creating, modifying, or granting permission on it (per
[Configure Unity AI Gateway endpoints](https://docs.databricks.com/aws/en/ai-gateway/configure-endpoints-beta)).
Do this manually in the workspace UI, **before** deploying the `all` target:

1. **Enable the preview**

   Account console → **Previews** → toggle **Unity AI Gateway** on. (Account
   admin only; skip if already enabled.)

2. **Create the endpoint**

   Workspace sidebar → **AI Gateway** → **Create Unity AI Gateway Endpoint**.

   - **Name**: `dais2026-ai-gateway` (or pick another name — pass it via
     `--params "AI_GATEWAY_ENDPOINT_NAME=<name>"` at deploy time). The job
     parameter defaults to `databricks-claude-sonnet-4-5`, so if you skip
     this param the agents target a foundation model by that name directly;
     set it to your governed endpoint name to get guardrails / audit.
   - **Primary model**: a foundation model whose tool-use is good. We use
     `databricks-claude-sonnet-4-5` to match the agents' default `LLM_MODEL`.
   - Click **Create**.

3. **Enable Inference Tables**

   Endpoint detail page → **Inference Tables** → **Edit** → enable.
   Point it at a UC schema you control (e.g. `<your-catalog>.ai_gateway`).
   This populates `<catalog>.ai_gateway.<endpoint>_payload` with one row
   per request — both allowed and blocked.

4. **Enable Usage Tracking**

   Endpoint detail page → **Usage Tracking** → **Edit** → enable. Per-request
   token counts land in `system.ai_gateway.usage` (account admins only —
   if your user can't query that table, ask an account admin to grant
   `SELECT ON SCHEMA system.ai_gateway` to your group).

5. **Configure Guardrails**

   Endpoint detail page → **Guardrails** → **Edit**. Enable:
   - **PII Detection** = **Block** (rejects requests containing SSNs, credit
     cards, etc — returns HTTP 400 before the LLM sees them).
   - **Jailbreak and Prompt Injection** = on.
   - **Unsafe Content** = on.

6. **Configure Rate Limits** (optional)

   Endpoint detail page → **Rate Limits** → **Edit**. Set a per-user QPM /
   TPM limit if you want to demo the burst-test in `MLflow.ipynb` →
   "Unity AI Gateway" section. Skip otherwise.

7. **Grant CAN_QUERY to each agent App's service principal** *(after first deploy)*

   Each Databricks App runs as its own service principal, and that SP — not
   `account users` — is what calls the gateway at request time. Unlike Model
   Serving endpoints, an App's SP is **not** automatically a member of
   `account users`, so a blanket `account users` grant will not cover it.
   Grant `CAN_QUERY` to each agent App's SP directly:

   - First deploy the `all` target (step 8) so the two agent Apps —
     `refund-agent-<catalog>` and `complaint-agent-<catalog>` — exist and
     have SPs. Find each SP with:

     ```python
     from databricks.sdk import WorkspaceClient
     w = WorkspaceClient()
     for app in ("refund-agent-<catalog>", "complaint-agent-<catalog>"):
         a = w.apps.get(app)
         print(app, "→ SP:", a.service_principal_client_id, a.service_principal_name)
     ```

   - Then: AI Gateway → `dais2026-ai-gateway` → **Permissions** →
     **Add user / group** → paste each App SP → **CAN_QUERY**.

   Why this is manual: v2 Beta gateway endpoints live on a separate API
   surface from regular serving endpoints, with no permissions API, and they
   can't be listed in `mlflow.models.resources.DatabricksServingEndpoint(...)`
   (it crashes with `NOT_FOUND: Dependent serving endpoint <gateway> does not
   exist`). The agent stages therefore probe the gateway with the *notebook*
   identity at deploy time and let the App validate it at startup with the
   *App SP* — but the App SP can only succeed once you've granted it CAN_QUERY
   here. If the grant is missing, the App's import-time gateway smoke-check
   fails and the App will not start.

   > **First-deploy ordering:** the agent Apps' startup validation needs this
   > grant, but the grant needs the Apps to exist. If the very first deploy
   > fails at App startup with a gateway-auth error, grant CAN_QUERY to the two
   > App SPs (now that they exist) and redeploy / restart the Apps.

8. **Deploy the `all` target with the gateway wired in**

   ```bash
   databricks bundle deploy -t all --var catalog=<your-catalog>
   databricks bundle run caspers -t all \
     --params "CATALOG=<your-catalog>,AI_GATEWAY_ENDPOINT_NAME=dais2026-ai-gateway"
   ```

   Pass the **same** catalog to `--var catalog` (deploy time) and
   `--params CATALOG` (run time) — see the `--var` vs `--params` note in
   `README.md` / `AGENTS.md`. The agent App names are baked at deploy time
   into the `REFUND_AGENT_APP_NAME` / `COMPLAINT_AGENT_APP_NAME` job params,
   so consumers (stream jobs, eval stages, ops dashboard) resolve the same
   names the agent stages deployed even if the two catalogs ever drift.

   At request time both agents refresh their OAuth bearer automatically:
   - Refund's `ChatDatabricks(use_ai_gateway=True)` authenticates through
     `databricks_openai.DatabricksOpenAI`, whose httpx `BearerAuth` calls
     `config.authenticate()` on every request — rotated M2M tokens are
     picked up with no per-request rebuild.
   - Complaint rebuilds its `dspy.LM` with a fresh bearer per request and
     applies it via `dspy.context(lm=...)` (thread-safe on the AgentServer
     worker threads — `dspy.configure()` is only called once at import).

> **Verify:** after deploy, send one request to each agent App
> (`refund-agent-<catalog>`, `complaint-agent-<catalog>`) — the ops
> dashboard's agent tiles, or the stream jobs, will do this — and check that
> rows appear in `<catalog>.ai_gateway.<endpoint>_payload`. Each should show
> HTTP 200 with non-zero token counts and the App SP in the `requester`
> column. If you see HTTP 400 "Invalid Token" or the Apps fail to start,
> re-check the CAN_QUERY grant in step 7.

### Pre-show warm-up

1. Open each runbook in the workspace and run its **pre-flight** cell
2. Click each app URL once (Ops Dashboard, Refund Manager) to dodge cold-start.
3. Click one sample question in each Genie space to warm `<catalog>-ops-warehouse` and `<catalog>-genie-warehouse`.
4. Open one Knowledge Assistant endpoint and the Supervisor endpoint to warm Model Serving.
5. Open all dashaboards
